# 06. Supabase pgvector 색인 구축

Steam 리뷰 원본에서 긍/부정 각 40만 건(총 80만 건)을 균형 샘플링해 정제한 뒤, Supabase pgvector(`reviews` 컬렉션)에 배치 단위로 적재하고 HNSW 인덱스를 생성한다.

**중요**: 임베딩 계산 + 80만 건 삽입은 로컬 환경에서 수 시간 걸릴 수 있다. 셀 4는 `output/06_pgvector_checkpoint.txt` 체크포인트를 기록하므로, 중단 후 재실행해도 마지막 지점부터 이어서 진행된다.

## 1. 환경 로드

`.env`에서 Supabase 연결 정보(`DATABASE_URL_DIRECT`)를 로드하고, 정제/색인 함수를 가져온다.

In [ ]:
import os, sys, time
from pathlib import Path
import pandas as pd
from datasets import load_dataset
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

load_dotenv()
from src.config import DATABASE_URL_DIRECT, RANDOM_SEED
from src.data.pipeline import clean_reviews
from src.rag.index import reset_index, add_batch, get_game_counts

N_PER_LABEL = 400_000
BATCH_SIZE = 5_000
CHECKPOINT_PATH = str(ROOT / "output" / "06_pgvector_checkpoint.txt")

## 2. 원본 로드 + 균형 샘플링

HF `ksang/steamreviews`에서 긍정/부정 각 40만 건씩 샘플링해 섞는다.

In [ ]:
ds = load_dataset("ksang/steamreviews", split="train")
df = ds.to_pandas()

pos = df[df["review_score"] == 1].sample(n=N_PER_LABEL, random_state=RANDOM_SEED)
neg = df[df["review_score"] == -1].sample(n=N_PER_LABEL, random_state=RANDOM_SEED)
sample = pd.concat([pos, neg]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
print(len(sample))  # 800000 이어야 함

## 3. 정제

`clean_reviews`로 텍스트 정제 + 라벨 변환. `app_name` 컬럼을 유지해 게임별 메타데이터로 사용한다.

In [ ]:
cleaned = clean_reviews(sample, "review_text", "review_score",
                        normalize=True, min_words=3, extra_cols=["app_name"])
cleaned["app_name"] = cleaned["app_name"].fillna("(unknown)")
print(len(cleaned))

## 4. 체크포인트 기반 배치 적재

중단 시 재실행하면 `CHECKPOINT_PATH`에 기록된 지점부터 이어서 진행한다. 최초 실행 시에만 `reset_index`로 컬렉션을 초기화한다.

In [ ]:
texts = cleaned["text"].tolist()
metadatas = [{"app_name": row.app_name, "label": int(row.label)}
             for row in cleaned.itertuples()]

start_batch = 0
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        start_batch = int(f.read().strip())
else:
    reset_index(DATABASE_URL_DIRECT)  # 최초 1회만 컬렉션 초기화

for s in range(start_batch, len(texts), BATCH_SIZE):
    e = min(s + BATCH_SIZE, len(texts))
    t0 = time.time()
    add_batch(texts[s:e], metadatas[s:e], DATABASE_URL_DIRECT)
    with open(CHECKPOINT_PATH, "w") as f:
        f.write(str(e))
    print(f"{e}/{len(texts)} ({time.time()-t0:.1f}s)")

## 5. HNSW 인덱스 생성

적재가 모두 끝난 뒤 1회만 실행한다.

In [ ]:
import psycopg

with psycopg.connect(DATABASE_URL_DIRECT) as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute("""
            CREATE INDEX IF NOT EXISTS langchain_pg_embedding_hnsw_idx
            ON langchain_pg_embedding
            USING hnsw (embedding vector_cosine_ops);
        """)
        cur.execute("ANALYZE langchain_pg_embedding;")
print("index created")

## 6. 검증 — 게임별 집계

최소 20건 이상 리뷰가 있는 게임 수와 상위 10개를 확인한다.

In [ ]:
counts = get_game_counts(min_count=20)
print(len(counts), counts[:10])